In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import io
import re

In [9]:

# === CONFIGURAÇÕES ===
# Caminho para seu CSV (pode ser o summary consolidado)
CSV_PATH = "./summary_results.csv"

In [10]:


# === LEITURA E LIMPEZA ===
# Tenta ler diretamente tratando vírgula decimal; se falhar, faz fallback lendo o arquivo e substituindo vírgula por ponto.
try:
    df = pd.read_csv(CSV_PATH, encoding='utf-8', decimal=",")
except Exception:
    with open(CSV_PATH, "r", encoding="utf-8") as f:
        content = f.read().replace(",", ".")
    df = pd.read_csv(io.StringIO(content))
# === PRÉ-PROCESSAMENTO ===
# Garante nomes consistentes
df.columns = [c.strip().replace('"', '') for c in df.columns]
# Remove espaços, corrige tipos numéricos
for col in ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Extrai linguagem e teste do nome do executável (ex: tc1_HS.exe)
def parse_test_lang(filename):
    base = str(filename).replace(".exe", "").replace(".EXE", "")
    parts = base.split("_")
    if len(parts) >= 2:
        test = "_".join(parts[:-1])
        lang = parts[-1]
        return test, lang
    return base, "Unknown"

df[["Test", "Language"]] = df["File"].apply(lambda f: pd.Series(parse_test_lang(f)))

# Extrai nível de otimização (o1, o2, o3...) do nome do assembly
df["Optimization"] = df["AsmFile"].apply(
    lambda x: re.search(r"o(\d+)", str(x)).group(1) if re.search(r"o(\d+)", str(x)) else "0"
)


In [11]:

# === DEBUG ===
print("\n=== DADOS FORMATADOS ===")
print(df.head(5))
print("\nLinguagens:", df["Language"].unique())
print("Testes:", df["Test"].unique())
print("Otimizacoes:", df["Optimization"].unique())

# === VISÃO GERAL ===
print("\n=== RESUMO DOS DADOS ===")
print(df.head())
print("\nLinguagens:", df['Language'].unique())
print("Testes:", df['Test'].unique())
print("Assembly Files:", df['AsmFile'].unique())



=== DADOS FORMATADOS ===
                  File  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  \
0  ops_complex1_HS.exe       185.76          2.027          3.051        0.87   
1  ops_complex1_HS.exe       162.11          2.027          3.047        1.73   
2  ops_complex1_HS.exe       186.68          2.027          3.051        0.87   
3  ops_complex1_HS.exe       197.44          2.027          3.047        1.74   
4  ops_complex1_HS.exe       174.30          2.027          3.043        0.87   

             AsmFile  AsmLines          Test Language Optimization  
0  ops_complex1_HS.s       729  ops_complex1       HS            0  
1  ops_complex1_HS.s       729  ops_complex1       HS            0  
2  ops_complex1_HS.s       729  ops_complex1       HS            0  
3  ops_complex1_HS.s       729  ops_complex1       HS            0  
4  ops_complex1_HS.s       729  ops_complex1       HS            0  

Linguagens: ['HS' 'C']
Testes: ['ops_complex1' 'tc1' 'tc2' 'tc3' 'tc4' '

In [12]:
# === FUNÇÃO DE COMPARAÇÃO (corrige eixo Y agregando por nível de otimização) ===
def compare_test2   (test_name):
    """Gera gráficos comparando C e Haskell para um teste específico (usa média por Optimization)."""
    subset = df[df["Test"] == test_name].copy()

    if subset.empty:
        print(f"⚠️ Nenhum dado encontrado para o teste '{test_name}'")
        return

    metrics = ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]
    metric_labels = {
        "ExecTime_ms": "Tempo de Execução (ms)",
        "MaxWorking_MB": "Memória Física Máx (MB)",
        "MaxPrivate_MB": "Memória Privada Máx (MB)",
        "AvgCPU_pct": "Uso de CPU (%)",
        "AsmLines": "Linhas Assembly"
    }

    # Garantir que Optimization seja ordenável (int) para um eixo X correto
    subset["OptLevel"] = pd.to_numeric(subset["Optimization"], errors="coerce").fillna(-1).astype(int)
    subset.sort_values("OptLevel", inplace=True)

    # Para cada métrica, plotar a média por (Optimization, Language) com barras (e erro padrão)
    for metric in metrics:
        agg = subset.groupby(["OptLevel", "Language"])[metric].agg(["mean", "std", "count"]).reset_index()
        if agg.empty:
            continue
        # exibir Optimization como string no eixo x na ordem numérica
        agg["Optimization"] = agg["OptLevel"].astype(str)

        fig = px.bar(
            agg,
            x="Optimization",
            y="mean",
            color="Language",
            barmode="group",
            error_y="std",
            labels={"mean": metric_labels[metric]},
            title=f"{test_name} — {metric_labels[metric]} (média por nível de otimização)",
        )
        fig.update_layout(
            template="plotly_white",
            font=dict(size=14),
            yaxis_title=metric_labels[metric],
            xaxis_title="Nível de Otimização"
        )
        fig.show()

    # Comparativo médio por linguagem (usado para resumo, radar e speed ratio)
    mean_data = subset.groupby("Language")[metrics].mean().reset_index()
    print(f"=== MÉDIAS — {test_name} ===")
    print(mean_data)
    print()

    # Radar Chart (perfil de desempenho) - normalização segura (evita divisão por zero)
    normalized = mean_data.copy()
    for col in metrics:
        mn = mean_data[col].min()
        mx = mean_data[col].max()
        if pd.isna(mn) or pd.isna(mx) or mx == mn:
            normalized[col] = 0.5  # valor neutro quando não há variação
        else:
            normalized[col] = (mean_data[col] - mn) / (mx - mn)

    fig_radar = go.Figure()
    for _, row in normalized.iterrows():
        fig_radar.add_trace(go.Scatterpolar(
            r=row[metrics].values,
            theta=[metric_labels[m] for m in metrics],
            fill="toself",
            name=row["Language"]
        ))
    fig_radar.update_layout(
        title=f"Perfil de Desempenho Normalizado — {test_name}",
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        template="plotly_white"
    )
    fig_radar.show()

    # Speed Ratio (quanto Haskell é mais lento/rápido que C) usando médias
    if {"C", "Haskell"}.issubset(set(mean_data["Language"])):
        c_data = mean_data.loc[mean_data["Language"] == "C"].iloc[0]
        hs_data = mean_data.loc[mean_data["Language"] == "Haskell"].iloc[0]

        # proteção contra divisão por zero
        if c_data["ExecTime_ms"] == 0:
            print(f"⚠️ ExecTime_ms média de C é zero para {test_name}, não é possível calcular ratio.")
        else:
            ratio = hs_data["ExecTime_ms"] / c_data["ExecTime_ms"]
            print(f"⚡ Speed Ratio ({test_name}): Haskell é {ratio:.2f}x o tempo de C.\n")


# === GERAR COMPARAÇÕES PARA TODOS OS TESTES ===
tests = df["Test"].unique()

for t in tests:
    compare_test2(t)

=== MÉDIAS — ops_complex1 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    175.63525        2.02670         0.4391     0.19575     127.0
1       HS    178.73450        2.13265         3.3229     1.08075     382.5



=== MÉDIAS — tc1 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C     174.2325       2.038425       0.434025     0.32600      80.5
1       HS     175.9620       2.043700       3.072200     0.93375     247.5



=== MÉDIAS — tc2 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    177.05200         2.0118       0.421800       0.239     91.25
1       HS    181.11325         2.0684       3.081575       0.978    206.50



=== MÉDIAS — tc3 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    176.90875       2.032500       0.425225       0.174    110.25
1       HS    182.22400       2.020475       3.050800       0.815    289.75



=== MÉDIAS — tc4 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    173.31425        2.01170         0.4217     0.15225     93.75
1       HS    177.06750        2.05935         3.0677     1.19475    324.50



...

=== MÉDIAS — min1 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    179.51975       2.011700       0.422000      0.2175     63.75
1       HS    176.85500       2.257975       3.360275      1.0110    160.25



=== MÉDIAS — min2 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    176.42750          2.012       0.422500     0.13050      75.0
1       HS    174.99025          2.060       3.059725     1.03175     173.5



=== MÉDIAS — min3 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    174.09525         2.0120          0.422     0.20650     85.25
1       HS    175.02600         2.0268          3.047     0.95625    182.75



=== MÉDIAS — min4 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C    175.47450       2.015125        0.42700       0.250      95.5
1       HS    183.48025       2.274775        3.59775       0.847     192.0



=== MÉDIAS — min5 ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        C     176.6285       2.011500        0.42320     0.36950   105.750
1       HS     175.9895       2.044375        3.05465     0.97775   197.825



In [13]:
# === CRIA COLUNA DE TIPO DE TESTE ===
def parse_test_type(test_name):
    if "tc" in test_name.lower():
        return "Time Complexity"
    elif "min" in test_name.lower():
        return "Minimal Complexity"
    elif "ops_complex" in test_name.lower():
        return "Operation Complexity"
    else:
        return "Other"

# ATENÇÃO: se necessário, adicione no fluxo principal:
df["TestType"] = df["Test"].apply(parse_test_type)

def compare_type_clean(test_type):
    # 1. Definição das métricas e rótulos
    metrics = ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]
    metric_labels = {
        "ExecTime_ms": "Tempo de Execução (ms)",
        "MaxWorking_MB": "Memória Física Máx (MB)",
        "MaxPrivate_MB": "Memória Privada Máx (MB)",
        "AvgCPU_pct": "Uso de CPU (%)",
        "AsmLines": "Linhas Assembly"
    }
    
    # 2. Filtra por tipo de teste
    subset = df[df["TestType"] == test_type]
    
    if subset.empty:
        print(f"⚠️ Nenhum dado para '{test_type}'")
        return

    # 3. AGREGAÇÃO: média por (Test, Language)
    agg_subset = subset.groupby(["Test", "Language"])[metrics].mean().reset_index()

    # Exibe um resumo indicando que é a MÉDIA (para deixar claro)
    print(f"=== AGREGADO (MÉDIA) — {test_type} ===")
    print(agg_subset.head())

    # 4. Geração dos Gráficos (com indicação explícita de "média")
    for metric in metrics:
        fig = px.bar(
            agg_subset,
            x="Test",
            y=metric,
            color="Language",
            barmode="group",
            text_auto="%d" if metric == "AsmLines" else ".2s", 
            title=f"✅ {test_type} — {metric_labels[metric]} (C vs Haskell) — média por teste",
            color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
        )
        
        fig.update_layout(
            template="plotly_white",
            font=dict(size=12),
            xaxis_title="Testes",
            # Marca explicitamente que o eixo Y representa a média
            yaxis_title=f"{metric_labels[metric]} (média)",
            xaxis_tickangle=-45,
            yaxis_type='linear' 
        )
        
        fig.show()


# === EXECUTA PARA TODOS OS TIPOS ===
types = df["TestType"].unique()
for t in types:
    compare_type_clean(t)


=== AGREGADO (MÉDIA) — Operation Complexity ===
           Test Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  \
0  ops_complex1        C    175.63525        2.02670         0.4391   
1  ops_complex1       HS    178.73450        2.13265         3.3229   

   AvgCPU_pct  AsmLines  
0     0.19575     127.0  
1     1.08075     382.5  


=== AGREGADO (MÉDIA) — Time Complexity ===
  Test Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  \
0  tc1        C    174.23250       2.038425       0.434025     0.32600   
1  tc1       HS    175.96200       2.043700       3.072200     0.93375   
2  tc2        C    177.05200       2.011800       0.421800     0.23900   
3  tc2       HS    181.11325       2.068400       3.081575     0.97800   
4  tc3        C    176.90875       2.032500       0.425225     0.17400   

   AsmLines  
0     80.50  
1    247.50  
2     91.25  
3    206.50  
4    110.25  


=== AGREGADO (MÉDIA) — Minimal Complexity ===
   Test Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  \
0  min1        C    179.51975       2.011700       0.422000     0.21750   
1  min1       HS    176.85500       2.257975       3.360275     1.01100   
2  min2        C    176.42750       2.012000       0.422500     0.13050   
3  min2       HS    174.99025       2.060000       3.059725     1.03175   
4  min3        C    174.09525       2.012000       0.422000     0.20650   

   AsmLines  
0     63.75  
1    160.25  
2     75.00  
3    173.50  
4     85.25  


In [14]:
# Variáveis e DataFrames (assumindo que 'df', 'metrics', e 'metric_labels' já estão definidos)
metrics = ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]
metric_labels = {
    "ExecTime_ms": "Tempo de Execução (ms)",
    "MaxWorking_MB": "Memória Física Máx (MB)",
    "MaxPrivate_MB": "Memória Privada Máx (MB)",
    "AvgCPU_pct": "Uso de CPU (%)",
    "AsmLines": "Linhas Assembly"
}

# --- 1. Gráfico de Linha: Desempenho (ExecTime) vs. Nível de Otimização, por Linguagem ---

# Objetivo: Mostrar como cada linguagem responde às otimizações.
# Agrega o tempo médio de execução por Nível de Otimização e Linguagem, 
# abrangendo todos os testes.
agg_time_opt = df.groupby(['Optimization', 'Language'])[metrics].mean().reset_index()

fig1 = px.line(
    agg_time_opt,
    x="Optimization",
    y="ExecTime_ms",
    color="Language",
    line_group="Language",
    markers=True,
    title="1. Impacto da Otimização no Tempo de Execução (Média Geral)",
)
fig1.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['ExecTime_ms'],
    xaxis_title="Nível de Otimização",
    font=dict(size=12)
)
fig1.show()


# --- 2. Gráfico de Linha: Complexidade do Código (AsmLines) vs. Nível de Otimização ---

# Objetivo: Mostrar se a otimização reduz ou aumenta o número de instruções geradas.
# Agrega AsmLines por Nível de Otimização e Linguagem.
agg_asm_opt = df.groupby(['Optimization', 'Language'])['AsmLines'].mean().reset_index()

fig2 = px.line(
    agg_asm_opt,
    x="Optimization",
    y="AsmLines",
    color="Language",
    line_group="Language",
    markers=True,
    title="2. Tendência do Número de Linhas Assembly vs. Nível de Otimização",
)
fig2.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['AsmLines'],
    xaxis_title="Nível de Otimização",
    font=dict(size=12)
)
fig2.show()


# --- 3. Gráfico de Linha: Uso de Memória (MaxWorking_MB) ao longo dos Testes ---

# Objetivo: Mostrar o consumo de recursos (Memória) para cada teste, em média.
# Agrega o consumo médio de memória por Teste e Linguagem.
agg_memory_test = df.groupby(['Test', 'Language'])['MaxWorking_MB'].mean().reset_index()

fig3 = px.line(
    agg_memory_test,
    x="Test",
    y="MaxWorking_MB",
    color="Language",
    line_group="Language",
    markers=True,
    title="3. Consumo de Memória (Média) ao Longo da Sequência de Testes",
)
fig3.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['MaxWorking_MB'],
    xaxis_title="Teste (Min1 a Min5 / Tc1 a TcX)",
    xaxis_tickangle=-45,
    font=dict(size=12)
)
fig3.show()

In [17]:
# --- VISUALIZACAO 4: VIOLIN PLOT (Distribuição do Tempo de Execução por Teste) ---

# Objetivo: Mostrar a variabilidade e a densidade do tempo de execução por teste.

fig4 = px.violin(
    df,
    y="ExecTime_ms",
    x="Test",
    color="Language",
    box=True,  # Inclui box plot dentro do violino
    points="all", # Mostra todos os pontos
    title="4. Distribuição do Tempo de Execução (ms) por Teste e Linguagem (Violin Plot)",
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)
fig4.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['ExecTime_ms'],
    xaxis_title="Teste",
    font=dict(size=12)
)
fig4.show()


In [18]:
# --- VISUALIZACAO 5 (separada): cria um scatter + trendline para cada Test individualmente ---

# Usa a variável `tests` já definida (lista/array de nomes de testes)
for test_name in tests:
    subset = df[df["Test"] == test_name]
    if subset.empty:
        print(f"⚠️ Sem dados para {test_name}")
        continue

    fig = px.scatter(
        subset,
        x="MaxWorking_MB",
        y="ExecTime_ms",
        color="Language",
        trendline="ols",
        title=f"5 — {test_name}: Tempo de Execução vs Memória (por linguagem)",
        labels={"ExecTime_ms": metric_labels["ExecTime_ms"], "MaxWorking_MB": metric_labels["MaxWorking_MB"]},
        color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
    )
    fig.update_layout(
        template="plotly_white",
        font=dict(size=12),
        height=420,
        width=760
    )
    fig.show()


In [41]:
# --- Preparação: Agregação por Média para o Mapa de Calor ---
heatmap_agg = df.groupby(['Test', 'Language', 'Optimization'])[metrics].mean().reset_index()

# --- VISUALIZACAO 6: HEATMAP - Linhas Assembly vs Teste & Otimização (Geral) ---
asm_heatmap = heatmap_agg.pivot_table(
    index='Test',
    columns=['Language', 'Optimization'],
    values='AsmLines'
)

# Flatten multiindex columns para legendas mais legíveis
asm_heatmap.columns = [f"{lang} — O{opt}" for lang, opt in asm_heatmap.columns]

fig6 = px.imshow(
    asm_heatmap,
    text_auto=True,
    color_continuous_scale="Plasma",
    aspect="auto",
    title="6. Mapa de Calor: Linhas Assembly (Complexidade) por Teste, Linguagem e Otimização",
    labels={'x': 'Linguagem — Nível de Otimização', 'y': 'Teste', 'color': f"{metric_labels['AsmLines']} (média)"}
)

# Ajustes nas legendas/labels/tick para ficar mais claro
fig6.update_layout(
    template="plotly_white",
    xaxis={'side': 'top'},
    height=550,
    font=dict(size=11)
)
fig6.update_xaxes(tickangle=-45)
# Personaliza hover e o título do colorbar (legenda de cor)
fig6.data[0].hovertemplate = 'Linguagem/Opt: %{x}<br>Teste: %{y}<br>Linhas Assembly (média): %{z}<extra></extra>'
fig6.data[0].colorbar.title.text = f"{metric_labels['AsmLines']} (média)"

fig6.show()


In [29]:
# --- VISUALIZACAO 7: SCATTER PLOT - Memória vs Tempo de Execução (Geral)
# Diferencia as linguagens pelo formato (símbolo) e mantém cor por TestType.

fig7 = px.scatter(
    df,
    x="MaxWorking_MB",
    y="ExecTime_ms",
    color="TestType",            # mantém cor por tipo de teste
    symbol="Language",           # diferencia linguagens pelo símbolo
    symbol_map={"C": "circle", "HS": "diamond"},
    size="AsmLines",             # tamanho pelo AsmLines
    hover_data=['Test', 'Language', 'Optimization'],
    title="7. Trade-off: Tempo de Execução vs. Memória (Tamanho = Linhas Assembly)",
    labels={"ExecTime_ms": metric_labels["ExecTime_ms"], "MaxWorking_MB": metric_labels["MaxWorking_MB"]},
    color_discrete_map={
        "Operation Complexity": "#636efa",
        "Time Complexity": "#EF553B",
        "Minimal Complexity": "#00cc96"
    }
)

fig7.update_traces(marker=dict(sizemin=5), selector=dict(mode='markers'))

fig7.update_layout(
    template="plotly_white",
    font=dict(size=12),
    legend_title_text="TestType / Language"  # legenda clara para cor (TestType) e símbolo (Language)
)
fig7.show()


In [22]:

# --- VISUALIZACAO 8: BOX PLOT - Variação do Uso de CPU (%) por Nível de Otimização ---

# Objetivo: Mostrar a distribuição (variação) do uso de CPU e identificar outliers.

fig8 = px.box(
    df,
    x="Optimization",
    y="AvgCPU_pct",
    color="Language",
    points="outliers", # Mostra apenas outliers (pontos extremos)
    title="8. Distribuição do Uso de CPU (%) por Otimização e Linguagem (Box Plot)",
    labels={"AvgCPU_pct": metric_labels["AvgCPU_pct"], "Optimization": "Nível de Otimização"},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)

fig8.update_layout(
    template="plotly_white",
    font=dict(size=12)
)
fig8.show()

In [33]:
# --- Agregação por Média para o 3D (Mais limpo) ---
agg_df = df.groupby(['Test', 'Language'])[metrics].mean().reset_index()

# --- VISUALIZAÇÃO 9: SCATTER PLOT 3D: Tempo, Memória e Linhas Assembly ---

# Objetivo: Visualizar como três métricas cruciais de desempenho se correlacionam.
# Espera-se que Haskell (H) e C (C) formem dois clusters distintos no espaço 3D.

fig9 = px.scatter_3d(
    agg_df,
    x='ExecTime_ms',
    y='MaxWorking_MB',
    z='AsmLines',
    color='Language', # Cor por Linguagem (C vs Haskell)
    symbol='Test',    # Forma do ponto por Teste
    title='9. Scatter Plot 3D: Relação entre Tempo, Memória e Linhas Assembly (Agregado)',
    labels={
        'ExecTime_ms': metric_labels['ExecTime_ms'],
        'MaxWorking_MB': metric_labels['MaxWorking_MB'],
        'AsmLines': metric_labels['AsmLines']
    }
)

fig9.update_layout(
    template="plotly_white",
    scene=dict(
        xaxis_title=metric_labels['ExecTime_ms'],
        yaxis_title=metric_labels['MaxWorking_MB'],
        zaxis_title=metric_labels['AsmLines']
    )
)
fig9.show()



In [32]:

# --- VISUALIZAÇÃO 10: SCATTER PLOT 3D: Tempo vs. Otimização vs. Teste ---

# Objetivo: Mostrar o impacto da otimização no Tempo de Execução para diferentes testes,
# usando o eixo Z para o tempo, e os eixos X e Y para as variáveis categóricas.
# Aqui usamos o DF completo (df), pois 'Optimization' é uma variável chave.

fig10 = px.scatter_3d(
    df,
    x='Test',
    y='Optimization',
    z='ExecTime_ms',
    color='Language',
    size='MaxWorking_MB', # O tamanho da bolha reflete o custo de memória
    title='10. Scatter Plot 3D: Tempo de Execução (Z) vs. Teste (X) e Otimização (Y)',
    labels={'ExecTime_ms': metric_labels['ExecTime_ms'], 'Optimization': 'Nível de Otimização', 'Test': 'Teste'},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)

fig10.update_layout(
    template="plotly_white",
    scene=dict(
        xaxis_title='Teste',
        yaxis_title='Otimização',
        zaxis_title=metric_labels['ExecTime_ms']
    )
)
fig10.show()

In [34]:
# --- VISUALIZAÇÃO 11: HISTOGRAMA - Distribuição do Tempo de Execução (ms) ---

# Objetivo: Mostrar a frequência de ocorrência dos tempos de execução.
# Usamos 'histfunc="count"' para contar as ocorrências em cada bin.

fig11 = px.histogram(
    df,
    x="ExecTime_ms",
    color="Language",
    marginal="box", # Adiciona um Box Plot na margem para ver a mediana e outliers
    opacity=0.7,
    title="11. Histograma: Distribuição de Frequência do Tempo de Execução (C vs Haskell)",
    labels={"ExecTime_ms": metric_labels["ExecTime_ms"]},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"},
    # O 'barmode' "overlay" é crucial para sobrepor os histogramas
    barmode="overlay" 
)

fig11.update_layout(
    template="plotly_white",
    yaxis_title="Contagem de Ocorrências",
    font=dict(size=12)
)
fig11.show()

In [35]:
# --- VISUALIZAÇÃO 12: HISTOGRAMA - Distribuição de Linhas Assembly ---

# Objetivo: Mostrar a frequência de diferentes tamanhos de código gerado.

fig12 = px.histogram(
    df,
    x="AsmLines",
    color="Language",
    marginal="rug", # Adiciona um gráfico de "tapete" na margem para ver a dispersão exata
    opacity=0.7,
    title="12. Histograma: Distribuição da Quantidade de Linhas Assembly (C vs Haskell)",
    labels={"AsmLines": metric_labels["AsmLines"]},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"},
    barmode="overlay"
)

fig12.update_layout(
    template="plotly_white",
    yaxis_title="Contagem de Ocorrências",
    font=dict(size=12)
)
fig12.show()

In [36]:
# Agregação necessária para o Heatmap (garante um valor médio Z por X/Y)
heatmap_agg = df.groupby(['Test', 'Optimization', 'Language'])[metrics].mean().reset_index()


# --- VISUALIZAÇÃO 13: HEATMAP - Tempo de Execução vs Teste & Otimização ---

# Objetivo: Mostrar onde o tempo de execução é mais alto/baixo.
# A cor representa o Tempo de Execução (ExecTime_ms).
# O gráfico é facetado pela Linguagem para comparação direta.

fig13 = px.density_heatmap(
    heatmap_agg,
    x="Test",
    y="Optimization",
    z="ExecTime_ms",
    histfunc="sum", # Sumariza o valor de ExecTime_ms
    facet_col="Language", # Separa C e Haskell
    title="13. Heatmap: Tempo de Execução (Média) por Teste e Otimização",
    labels={'ExecTime_ms': metric_labels['ExecTime_ms'], 'Optimization': 'Nível de Otimização'},
    color_continuous_scale=px.colors.sequential.Inferno
)

fig13.update_layout(
    template="plotly_white",
    xaxis={'side': 'bottom'},
    height=400,
    font=dict(size=12)
)
fig13.show()



In [ ]:

# --- VISUALIZAÇÃO 14: HEATMAP - Correlação de Métricas (C vs Haskell Lado a Lado) ---

# Objetivo: Comparar a correlação interna das métricas entre as duas linguagens.

# Agrega por Language e calcula a matriz de correlação em um Loop
corr_data = []
for lang in df['Language'].unique():
    subset = df[df['Language'] == lang][metrics]
    corr_matrix = subset.corr(method='pearson').reset_index().rename(columns={'index': 'Métrica'})
    corr_matrix = pd.melt(corr_matrix, id_vars='Métrica', var_name='Métrica 2', value_name='Correlação')
    corr_matrix['Language'] = lang
    corr_data.append(corr_matrix)

final_corr_df = pd.concat(corr_data)

fig14 = px.imshow(
    final_corr_df,
    x="Métrica",
    y="Métrica 2",
    color="Correlação",
    facet_col="Language",
    text_auto=".2f",
    color_continuous_scale=px.colors.diverging.RdBu,
    range_color=[-1, 1], # Fixa a escala de cor de -1 a 1 para comparação justa
    title="14. Heatmap: Correlação de Métricas (C vs Haskell)",
)

fig14.update_layout(
    template="plotly_white",
    xaxis={'side': 'top'},
    height=500,
    font=dict(size=10)
)
fig14.show()